# Query Tester (Semantic Scholar + OpenAlex)

This notebook is intentionally small and **only** meant for quickly iterating on query strings / filters and inspecting what comes back.

It reads credentials from the project-root `.env`:
- `SEMANTICSCHOLAR_API_KEY`
- `OPENALEX_API_KEY`
- `OPENALEX_EMAIL` (optional; sent as `mailto=`)

For each source it shows:
- **Top cited** (20)
- **Random sample** (20)
- **A few raw JSON examples** (random)

Fields emphasized: **title**, **year**, **DOI**, **citation count**, **language**, **has abstract**, **abstract preview**.


In [2]:
# Setup + helpers (run once)

import json
import os
import random
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import requests

try:
    import pandas as pd  # optional but recommended for nice tables
except Exception:
    pd = None

try:
    from IPython.display import display
except Exception:
    display = None


def _project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "sources-v2" else cwd


PROJECT_ROOT = _project_root()
DOTENV_PATH = PROJECT_ROOT / ".env"


def load_dotenv_simple(path: Path) -> None:
    """Minimal .env loader (no external deps)."""
    if not path.exists():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()
        if value and value[0] == value[-1] and value[0] in ('"', "'"):
            value = value[1:-1]
        os.environ.setdefault(key, value)


load_dotenv_simple(DOTENV_PATH)

SEMANTICSCHOLAR_API_KEY = (os.getenv("SEMANTICSCHOLAR_API_KEY") or "").strip()
OPENALEX_API_KEY = (os.getenv("OPENALEX_API_KEY") or "").strip()
OPENALEX_EMAIL = (os.getenv("OPENALEX_EMAIL") or "").strip()

DEFAULT_TIMEOUT_S = 30
ABSTRACT_PREVIEW_CHARS = 350
RAW_JSON_EXAMPLES = 3


def truncate(text: Optional[str], n: int = ABSTRACT_PREVIEW_CHARS) -> str:
    s = (text or "").strip()
    if len(s) <= n:
        return s
    return s[: max(0, int(n) - 1)] + "…"


def normalize_doi(doi: Optional[str]) -> Optional[str]:
    if not doi:
        return None
    s = doi.strip()
    s = s.replace("https://doi.org/", "").replace("http://doi.org/", "")
    if s.lower().startswith("doi:"):
        s = s.split(":", 1)[1].strip()
    return s or None


def pretty_json(obj: Any, max_chars: int = 20_000) -> str:
    s = json.dumps(obj, ensure_ascii=False, indent=2)
    if len(s) <= max_chars:
        return s
    return s[:max_chars] + "\n…(truncated)…"


def request_json(
    url: str,
    *,
    params: Optional[Dict[str, Any]] = None,
    headers: Optional[Dict[str, str]] = None,
    timeout_s: int = DEFAULT_TIMEOUT_S,
    max_retries: int = 6,
    backoff: float = 1.7,
) -> Tuple[Dict[str, Any], requests.Response]:
    """GET JSON with basic retry/backoff (handles 429)."""
    params = params or {}
    headers = headers or {}

    last_exc: Optional[Exception] = None
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=timeout_s)
            if resp.status_code == 429:
                retry_after = resp.headers.get("Retry-After")
                sleep_s = float(retry_after) if retry_after else (backoff**attempt)
                time.sleep(min(sleep_s, 30))
                continue
            resp.raise_for_status()
            return resp.json(), resp
        except Exception as e:
            last_exc = e
            time.sleep(min(backoff**attempt, 12))

    assert last_exc is not None
    raise last_exc


def show_rows(rows: List[Dict[str, Any]], *, columns: List[str], max_rows: int = 20):
    if not rows:
        print("(no rows)")
        return None

    if pd is None:
        print("Tip: install pandas for nicer tables: %pip install pandas")
        shown = rows[: int(max_rows)]

        def _cell(v: Any) -> str:
            s = "" if v is None else str(v)
            s = s.replace("\n", " ")
            return truncate(s, 120)

        widths = {c: len(str(c)) for c in columns}
        table = []
        for r in shown:
            row = {c: _cell(r.get(c)) for c in columns}
            table.append(row)
            for c in columns:
                widths[c] = max(widths[c], len(row[c]))

        header = " | ".join(str(c).ljust(widths[c]) for c in columns)
        sep = "-+-".join("-" * widths[c] for c in columns)
        print(header)
        print(sep)
        for row in table:
            print(" | ".join(row[c].ljust(widths[c]) for c in columns))
        return None

    df = pd.DataFrame(rows)
    for c in columns:
        if c not in df.columns:
            df[c] = None
    df = df[columns]

    pd.set_option("display.max_colwidth", 140)
    pd.set_option("display.width", 140)

    if display is not None:
        display(df.head(max_rows))
    else:
        print(df.head(max_rows).to_string(index=False))
    return df


def openalex_abstract_from_inverted_index(inv: Optional[Dict[str, List[int]]]) -> Optional[str]:
    """Convert OpenAlex abstract_inverted_index into plain text."""
    if not inv:
        return None

    max_pos = -1
    for _, positions in inv.items():
        for p in positions:
            if isinstance(p, int) and p > max_pos:
                max_pos = p

    if max_pos < 0:
        return None

    words: List[Optional[str]] = [None] * (max_pos + 1)
    for word, positions in inv.items():
        for p in positions:
            if isinstance(p, int) and 0 <= p <= max_pos:
                words[p] = word

    text = " ".join(w for w in words if w)
    return text.strip() or None


print("Loaded .env:")
print("- SEMANTICSCHOLAR_API_KEY:", "yes" if SEMANTICSCHOLAR_API_KEY else "no")
print("- OPENALEX_API_KEY:", "yes" if OPENALEX_API_KEY else "no")
print("- OPENALEX_EMAIL:", "yes" if OPENALEX_EMAIL else "no")


Loaded .env:
- SEMANTICSCHOLAR_API_KEY: yes
- OPENALEX_API_KEY: yes
- OPENALEX_EMAIL: no


## Semantic Scholar (S2) — paper search

Notes:
- This uses the Graph API `paper/search` endpoint.
- The Graph API does **not** support a `language` field in `fields=` (so `language` will be blank here).
- Random sampling fetches a few pages at random offsets (capped by `S2_MAX_RANDOM_OFFSET`).


In [35]:
# --- S2: edit these ---

S2_QUERY = '+(("Late Antiquity" | "Western Roman Empire" | "urban decline" | taxation | coinage)) +("state revenues" | currency | "military financing" | "coinage" | "public finance")'  # TODO: replace with your query

# Keep this small: we're only pulling the fields you care about (+ url/paperId for debugging)
S2_FIELDS = "paperId,title,year,citationCount,externalIds,abstract,url"

S2_TOP_N = 20
S2_RANDOM_N = 20

# Random sampling: fetch a few pages at random offsets to build a pool, then sample from it.
S2_RANDOM_POOL_REQUESTS = 6
S2_RANDOM_PAGE_SIZE = 100

# Avoid deep pagination limits by capping offsets.
S2_MAX_RANDOM_OFFSET = 9000
S2_SLEEP_BETWEEN_REQUESTS_SEC = 1.1

S2_RANDOM_SEED = None  # set e.g. 42 to make the random sample reproducible


In [36]:
S2_SEARCH_URL = "https://api.semanticscholar.org/graph/v1/paper/search"


def s2_search_page(
    *,
    query: str,
    fields: str,
    limit: int,
    offset: int = 0,
    sort: Optional[str] = None,
) -> Dict[str, Any]:
    if not SEMANTICSCHOLAR_API_KEY:
        raise RuntimeError("Missing SEMANTICSCHOLAR_API_KEY (set it in .env)")

    params: Dict[str, Any] = {
        "query": query,
        "fields": fields,
        "limit": int(limit),
        "offset": int(offset),
    }
    if sort:
        params["sort"] = sort

    headers = {"x-api-key": SEMANTICSCHOLAR_API_KEY}
    payload, _ = request_json(S2_SEARCH_URL, params=params, headers=headers)
    return payload


def s2_to_row(paper: Dict[str, Any]) -> Dict[str, Any]:
    ext = paper.get("externalIds") or {}
    abstract = paper.get("abstract")
    return {
        "title": paper.get("title"),
        "year": paper.get("year"),
        "doi": normalize_doi(ext.get("DOI")),
        "citation_count": paper.get("citationCount"),
        "language": None,
        "has_abstract": bool(abstract),
        "abstract_preview": truncate(abstract),
        "paperId": paper.get("paperId"),
        "url": paper.get("url"),
    }


print("S2 query:", S2_QUERY)

print("\n--- S2: top cited (sorted by citationCount desc) ---")
s2_top_payload = s2_search_page(
    query=S2_QUERY,
    fields=S2_FIELDS,
    limit=int(S2_TOP_N),
    offset=0,
    sort="citationCount:desc",
)
s2_total = s2_top_payload.get("total")
print("Total matches (S2):", s2_total)
s2_top_papers = s2_top_payload.get("data") or []
s2_top_rows = [s2_to_row(p) for p in s2_top_papers]
show_rows(
    s2_top_rows,
    columns=["title", "year", "doi", "citation_count", "language", "has_abstract", "abstract_preview", "paperId"],
    max_rows=int(S2_TOP_N),
)

print("\n--- S2: random sample ---")
if S2_RANDOM_SEED is not None:
    random.seed(int(S2_RANDOM_SEED))

pool: List[Dict[str, Any]] = []

try:
    total_int = int(s2_total) if s2_total is not None else 0
except Exception:
    total_int = 0

max_offset = max(0, total_int - int(S2_RANDOM_PAGE_SIZE)) if total_int else 0
max_offset = min(max_offset, int(S2_MAX_RANDOM_OFFSET))

offsets = [0] if max_offset == 0 else [random.randint(0, max_offset) for _ in range(int(S2_RANDOM_POOL_REQUESTS))]

for i, off in enumerate(offsets):
    payload = s2_search_page(
        query=S2_QUERY,
        fields=S2_FIELDS,
        limit=int(S2_RANDOM_PAGE_SIZE),
        offset=int(off),
        sort=None,
    )
    pool.extend(payload.get("data") or [])
    if i < len(offsets) - 1:
        time.sleep(float(S2_SLEEP_BETWEEN_REQUESTS_SEC))

# De-dupe by paperId
uniq: Dict[str, Dict[str, Any]] = {}
for p in pool:
    pid = p.get("paperId")
    if pid and pid not in uniq:
        uniq[pid] = p
pool = list(uniq.values())

print("Pool size fetched (deduped):", len(pool))
print("Pool has abstract:", sum(1 for p in pool if p.get("abstract")), "/", len(pool))

s2_rand_sample = random.sample(pool, k=min(int(S2_RANDOM_N), len(pool))) if pool else []
s2_rand_rows = [s2_to_row(p) for p in s2_rand_sample]
show_rows(
    s2_rand_rows,
    columns=["title", "year", "doi", "citation_count", "language", "has_abstract", "abstract_preview", "paperId"],
    max_rows=int(S2_RANDOM_N),
)

print("\n--- S2: raw JSON examples (random) ---")
for i, paper in enumerate(s2_rand_sample[: int(RAW_JSON_EXAMPLES)], start=1):
    print(f"\n[S2 raw #{i}] paperId={paper.get('paperId')}")
    print(pretty_json(paper))


S2 query: +(("Late Antiquity" | "Western Roman Empire" | "urban decline" | taxation | coinage)) +("state revenues" | currency | "military financing" | "coinage" | "public finance")

--- S2: top cited (sorted by citationCount desc) ---
Total matches (S2): 0
(no rows)

--- S2: random sample ---
Pool size fetched (deduped): 0
Pool has abstract: 0 / 0
(no rows)

--- S2: raw JSON examples (random) ---


## OpenAlex — works

Notes:
- Citation count is `cited_by_count`.
- Year is `publication_year`.
- Abstract (if present) comes as `abstract_inverted_index` and is reconstructed here.
- For random results, this uses OpenAlex' `sample=` parameter.


In [33]:
# --- OpenAlex: edit these ---

# Use either/both:
OPENALEX_SEARCH = '"Late Antiquity" AND (taxation OR coinage OR "military financing" OR "urban decline" OR "state revenues") '  # free-text search
OPENALEX_FILTER = ""  # e.g. "from_publication_date:2020-01-01,has_doi:true"

OPENALEX_TOP_N = 20
OPENALEX_RANDOM_N = 20

# Keep this tight so responses stay fast
OPENALEX_SELECT = (
    "id,title,publication_year,doi,language,cited_by_count,abstract_inverted_index"
)


In [34]:
OPENALEX_WORKS_URL = "https://api.openalex.org/works"


def openalex_get_works(params: Dict[str, Any]) -> Dict[str, Any]:
    base: Dict[str, Any] = {}
    if OPENALEX_API_KEY:
        base["api_key"] = OPENALEX_API_KEY
    if OPENALEX_EMAIL:
        base["mailto"] = OPENALEX_EMAIL
    if OPENALEX_SELECT:
        base["select"] = OPENALEX_SELECT

    merged = {**base, **{k: v for k, v in params.items() if v not in (None, "")}}
    headers = {"User-Agent": "instantpaper-query-tester"}
    payload, _ = request_json(OPENALEX_WORKS_URL, params=merged, headers=headers)
    return payload


def openalex_to_row(work: Dict[str, Any]) -> Dict[str, Any]:
    abstract = openalex_abstract_from_inverted_index(work.get("abstract_inverted_index"))
    return {
        "title": work.get("title"),
        "year": work.get("publication_year"),
        "doi": normalize_doi(work.get("doi")),
        "citation_count": work.get("cited_by_count"),
        "language": work.get("language"),
        "has_abstract": bool(abstract),
        "abstract_preview": truncate(abstract),
        "id": work.get("id"),
    }


print("OpenAlex search:", OPENALEX_SEARCH)
print("OpenAlex filter:", OPENALEX_FILTER or "(none)")

print("\n--- OpenAlex: top cited (sorted by cited_by_count desc) ---")
oa_top_payload = openalex_get_works(
    {
        "search": OPENALEX_SEARCH,
        "filter": OPENALEX_FILTER,
        "sort": "cited_by_count:desc",
        "per-page": int(OPENALEX_TOP_N),
    }
)
oa_top_meta = oa_top_payload.get("meta") or {}
oa_total = oa_top_meta.get("count")
print("Total matches (OpenAlex):", oa_total)
oa_top_works = oa_top_payload.get("results") or []
oa_top_rows = [openalex_to_row(w) for w in oa_top_works]
show_rows(
    oa_top_rows,
    columns=["title", "year", "doi", "citation_count", "language", "has_abstract", "abstract_preview", "id"],
    max_rows=int(OPENALEX_TOP_N),
)

print("\n--- OpenAlex: random sample (uses sample=) ---")
oa_rand_payload = openalex_get_works(
    {
        "search": OPENALEX_SEARCH,
        "filter": OPENALEX_FILTER,
        "sample": int(OPENALEX_RANDOM_N),
        "per-page": int(OPENALEX_RANDOM_N),
    }
)
oa_rand_meta = oa_rand_payload.get("meta") or {}
print("Total matches (OpenAlex):", oa_total)
oa_rand_works = oa_rand_payload.get("results") or []
print("Random results returned:", len(oa_rand_works))
print(
    "Random has abstract:",
    sum(1 for w in oa_rand_works if w.get("abstract_inverted_index")),
    "/",
    len(oa_rand_works),
)

oa_rand_rows = [openalex_to_row(w) for w in oa_rand_works]
show_rows(
    oa_rand_rows,
    columns=["title", "year", "doi", "citation_count", "language", "has_abstract", "abstract_preview", "id"],
    max_rows=int(OPENALEX_RANDOM_N),
)

print("\n--- OpenAlex: raw JSON examples (random) ---")
for i, work in enumerate(oa_rand_works[: int(RAW_JSON_EXAMPLES)], start=1):
    print(f"\n[OpenAlex raw #{i}] id={work.get('id')}")
    print(pretty_json(work))


OpenAlex search: "Late Antiquity" AND (taxation OR coinage OR "military financing" OR "urban decline" OR "state revenues") 
OpenAlex filter: (none)

--- OpenAlex: top cited (sorted by cited_by_count desc) ---
Total matches (OpenAlex): 1995


,title,year,doi,citation_count,language,has_abstract,abstract_preview,id
0,The hermeneutic style in tenth-century Anglo-Latin literature,1975,10.1017/s0263675100002726,285,en,True,Some twenty years ago the late Professor Alistair Campbell observed that there were two broad stylistic traditions of Anglo-Latinity: th...,https://openalex.org/W1997981416
1,"Housing the Stranger in the Mediterranean World: Lodging, Trade, and Travel in Late Antiquity and the Middle Ages",2009,10.1017/cbo9780511496264,244,en,True,"The Greek pandocheion, Arabic funduq, and Latin fundicum (fondaco) were ubiquitous in the Mediterranean sphere for nearly two millennia....",https://openalex.org/W630029574
2,Ethnic Constructs in Antiquity : The Role of Power and Tradition,2009,10.5117/9789089640789,220,en,True,This volume explores the theme of ethnicity and ethnogenesis in societies of the ancient world. Its starting point is the current view i...,https://openalex.org/W1536656927
3,Psychology as a Historical Science,2020,10.1146/annurev-psych-082820-111436,189,en,True,"Psychology has traditionally seen itself as the science of universal human cognition, but it has only recently begun seriously grappling...",https://openalex.org/W3092131469
4,Crises and the Roman Empire,2007,10.1163/ej.9789004160507.i-448,171,en,True,"This volume presents the proceedings of the seventh workshop of the international thematic network Impact of Empire, which concentrates ...",https://openalex.org/W1548326554
5,The Justinianic Plague: An inconsequential pandemic?,2019,10.1073/pnas.1903797116,166,en,True,Existing mortality estimates assert that the Justinianic Plague (circa 541 to 750 CE) caused tens of millions of deaths throughout the M...,https://openalex.org/W2991293248
6,The Economic Origins of the Territorial State,2016,10.1017/s0020818316000308,161,en,True,Abstract This paper challenges the long-standing belief that changes in patterns of war and war making caused the emergence of large ter...,https://openalex.org/W2247233429
7,Resetting the Urban Network: 117-2012,2016,10.1111/ecoj.12424,157,en,True,"Do fixed geographic features such as coastlines and rivers determine town locations, or can historical events trap towns in unfavourable...",https://openalex.org/W2096528289
8,"Rats, Communications, and Plague: Toward an Ecological History",2003,10.1162/002219503322645439,157,en,True,"During the last twenty years, archaeozoological research has significantly transformed the picture of the black rat (rattus rattus) in c...",https://openalex.org/W2065118693
9,Archaeoseismology: Methodological issues and procedure,2006,10.1007/s10950-006-9027-x,140,en,True,Archaeoseismic research contributes important data on past earthquakes. A limitation of the usefulness of archaeoseismology is due to th...,https://openalex.org/W2071097350



--- OpenAlex: random sample (uses sample=) ---
Total matches (OpenAlex): 1995
Random results returned: 20
Random has abstract: 17 / 20


,title,year,doi,citation_count,language,has_abstract,abstract_preview,id
0,Medieval Georgian Coinage,2024,10.52214/uw.v32i.12671,0,en,True,"This article surveys the medieval coinage minted on Georgian lands to the Mongol age. Aimed at college-level pedagogy, it underscores th...",https://openalex.org/W4405400517
1,Classical Heritage and European Identities: The Imagined Geographies of Danish Classicism,2019,10.4324/9780429455179,2,en,True,Classical Heritage and European Identities examines how the heritages of classical antiquity have been used to construct European identi...,https://openalex.org/W2918159054
2,"Rhetoric, Spatiality, and the First-Century Synagogue: The Description and Narrative Use of Jewish Institutions in the Works of Flavius ...",2015,None,0,en,True,The information about the first-century synagogue provided by Flavius Josephus must be handled with care when used in historical reconst...,https://openalex.org/W7115822779
3,"Plagues, Wages, and Economic Change in the Islamic Middle East, 700–1500",2014,10.1017/s0022050714000072,86,en,True,This study establishes long-term trends in the purchasing power of the wages of unskilled workers and develops estimates for GDP per cap...,https://openalex.org/W2066946319
4,"Celt and Saxon : interaction in pre-Viking Britain, c.600-800",2023,10.25959/23233178,0,en,True,"The thesis aims to present a history of the interaction between Anglo-Saxons and Celts in pre-Viking Britain, c.600-800. The study is or...",https://openalex.org/W71245356
5,Russian Energy Policy,2011,10.3929/ethz-a-006592621,2,en,True,ISSN:1863-0421,https://openalex.org/W2475065017
6,"Classical, not Classicistic Thoughts on the origins of «Classicizing Roman Sculpture»",2002,10.12681/eul.32740,0,it,True,"Classico, non classicista: riflessioni sulle origini della cosiddetta «scultura romana classicistica» In questo articolo è affrontata la...",https://openalex.org/W4317737877
7,Image and Ornament in the Early Medieval West,2023,10.1017/9781009207768,1,en,True,Scholarship often treats the post-Roman art produced in central and north-western Europe as representative of the pagan identities of th...,https://openalex.org/W4316012006
8,Preliminary Material,2021,10.1163/9789004501904_001,0,en,False,,https://openalex.org/W4387851430
9,Performing Geography in Global History1,2016,10.4000/eastafrica.321,0,fr,True,"Global history is an innovative approach which permits the integration of research on under-studied subjects, and places, without being ...",https://openalex.org/W2971246667



--- OpenAlex: raw JSON examples (random) ---

[OpenAlex raw #1] id=https://openalex.org/W4405400517
{
  "id": "https://openalex.org/W4405400517",
  "title": "Medieval Georgian Coinage",
  "publication_year": 2024,
  "doi": "https://doi.org/10.52214/uw.v32i.12671",
  "language": "en",
  "cited_by_count": 0,
  "abstract_inverted_index": {
    "This": [
      0
    ],
    "article": [
      1
    ],
    "surveys": [
      2
    ],
    "the": [
      3,
      11,
      20,
      35,
      39,
      45,
      49,
      53,
      74,
      79,
      89,
      96,
      107,
      121,
      154,
      161,
      164,
      167,
      173,
      179,
      204,
      207
    ],
    "medieval": [
      4
    ],
    "coinage": [
      5,
      123,
      147
    ],
    "minted": [
      6
    ],
    "on": [
      7,
      120,
      145,
      172,
      178
    ],
    "Georgian": [
      8,
      111,
      131
    ],
    "lands": [
      9
    ],
    "to": [
      10,
      70,
      141,
  